# Žmogus-procese: Veiksmų prieš pradedant patikros, rizikos pasiskirstymas ir audito žurnalai

Šio pamokos README pristato Žmogaus-procese (Human-in-the-Loop) modelį su trumpu fragmentu, kuriame vartotojui po agente atsakymo pateikiama galimybė `APPROVE` arba `REJECT`. Šis modelis yra geras pradinis taškas, tačiau realiose HITL implementacijose dažnai reikalingi dar trys papildomi elementai:

1. **Veiksmo prieš pradedant patikros** vartai, kurie veikia **prieš** agentui atliekant rizikingą žingsnį, kad būtų kontroliuojama kaina, negrįžtamumas ir vėlinimas.
2. **Rizikos pasiskirstymas**, kad mažos rizikos veiksmai būtų atliekami automatiškai, vidutinės rizikos veiksmai būtų patvirtinami paketais, o tik didelės rizikos veiksmai būtų sustabdyti žmogaus.
3. **Audito žurnalas ir peržiūrėjimo ciklas**, kad kiekvienas vartų sprendimas būtų įrašytas kaip JSONL, o atmetimas vėl inicijuotų agentą su struktūrizuota priežastimi, o ne tiesiog spausdintų `Revising...`.

Ši užduotis sukuria kiekvieną iš šių elementų, naudodama tuos pačius pradmenis kaip ir `06-system-message-framework.ipynb`. Ji veikia pilnai automatiniu režimu `DEMO_MODE = True` (nereikia interaktyvaus įvesties) arba su tikrais `input()` užklausomis, kai `DEMO_MODE = False`. Pastaba: DEMO_MODE režimu trečio tikslo bandymas aprašytas scenarijinėmis priemonėmis, tad ciklo mechanika matoma pilnai. Tikras peržiūromis pagrįstas pakartotinis klasifikavimas reikalauja `DEMO_MODE = False` ir operatoriaus.

**Už šio pamoko ribų (valdomi kituose pamokose):** autentifikacija ir prieigos kontrolė (Pamoka 06 README grėsmė 2), įrankių kvietimo tarpinė programinė įranga (Pamoka 14 MAF giluminė analizė), kelių agentų diskusijų modeliai.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Šablonas 1: Veiksmo prieš vartus

README HITL fragmentas pirmiausia kviečia agentą, tada prašo vartotojo patvirtinti išvestį. Tai yra **po veiksmo** srautas. Agentas jau įvykdytas, taigi LLM kvietimo kaina jau sumokėta, ir bet koks šalutinis poveikis (išsiųstas el. laiškas, įrašytas duomenų bazės eilutė, paskelbtas komentaras) jau įvyko.

**Prieš veiksmo** srautas įterpia vartus prieš agentui atliekant rizikingą žingsnį. Agentas siūlo veiksmą, vartai nusprendžia, ar vykdyti, ir tik gavus patvirtinimą įvyksta šalutinis poveikis.

| Aspektas | Po veiksmo patvirtinimas (README fragmentas) | Prieš veiksmo vartai (šis užrašų knygelė) |
|---|---|---|
| Kada vykdomas patvirtinimas? | Po to, kai agentas sukūrė išvestį | Prieš bet kokiam šalutiniam poveikiui vykstant |
| LLM kaina atmetus | Jau sumokėta | Sumokėta tik už pasiūlymą, ne už veiksmą |
| Nepakeičiami šalutiniai poveikiai | Gali būti (veiksmas jau įvyko) | Užkirsta kelias |
| Auditavimo aiškumas | Patvirtinimas yra spausdintas tekstas | Patvirtinimas yra JSON įrašas su laiko žyma, veiksmu, priežastimi |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Modelis 2: Rizikos sluoksniavimas

Ne kiekvienas veiksmas reikalauja žmogaus patvirtinimo. Tik skaitymui skirtas užklausimas viešajam API turi kitokią reikšmę nei siųsti el. laišką klientui. Abu tokio pat lygio traktuoti veltui švaisto operatoriaus dėmesį ir sulėtina agentą.

Paprastas 3 sluoksnių modelis:

| Sluoksnis | Pavyzdžiai | Patvirtinimo eiga |
|---|---|---|
| `žemas` (tik skaitymui) | Ieškoti žinių bazėje, ieškoti skrydžių variantų, gauti viešą tinklalapį | Automatinis vykdymas, registruojama audito tikslais |
| `vidutinis` (pigesnis mutacijos veiksmas) | Talpinti rezultatą, didinti skaitiklį, suplanuoti priminimą | Automatinis vykdymas, bet apžvelgiama kasdien partijos principu |
| `aukštas` (išorinės sąsajos arba negrįžtami veiksmai) | Siųsti el. laišką, nuskaičiuoti pinigus, paskelbti viešame kanale | Blokuojama kol nebus žmogaus patvirtinimo |

Tai vienas sluoksniavimo būdas. Produkcinės sistemos dažnai naudoja tikslesnius sluoksnius (pvz., AWS IAM leidimų lygius, pagal roles pagrįstus prieigos sluoksnius). Žemiau esanti 3 sluoksnių versija yra mažiausia naudinga versija agentui, maišant skaitomus ir šalutinius veiksmus.

Žemiau esantis klasifikatorius naudoja raktažodžių heuristiką, kad demonstracija būtų deterministinė ir pigi. Produkcinėje sistemoje jūs jį pakeistumėte mokomu klasifikatoriumi arba politikos varikliu.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Modelis 3: Audito žurnalas ir peržiūros ciklas

`print("Response approved.")` nėra audito žurnalas. Dėl pasitikėjimo kiekvienas vartų sprendimas turėtų būti įrašomas kaip struktūrinis įvykis, kurį vėliau galite užklausinėti, paleisti iš naujo arba pridėti prie incidento peržiūros.

Du elementai:

1. **Tik pridėjimo JSONL formatas.** Viena eilutė kiekvienam sprendimui, su laiko žyma, veiksmu, lygiu, sprendimu, priežastimi. Lengva ieškoti, lengva vėliau perduoti į tikrą žurnalo saugyklą.
2. **Peržiūros ciklas atmetimo atveju.** Kai vartai grąžina `deny`, agentas pakartotinai užduoda klausimą pats sau su atmetimo priežastimi kontekste, kad kitas pasiūlymas galėtų išvengti problemos.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Papildomi ištekliai

Keletas kitų viešų projektų įgyvendina šių HITL šablonų variacijas. Palyginkite požiūrius, kad surastumėte, kas tinka jūsų sistemai:

- **LangChain** žmogaus cikle įrankių apvyniojimai ([dokumentacija](https://python.langchain.com/docs/integrations/tools/human_tools)): įrankių apvyniojimai, kurie sustabdo vykdymą, kad būtų įvestas žmogaus indėlis.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentacija](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ pertvarkytas): naudoja agento vaidmenį, skirtą žmogui atstovauti daugiaagentinėse pokalbių situacijose.
- **Microsoft Agent Framework (MAF)** funkcijų kvietimo tarpinė programinė įranga ([dokumentacija](https://learn.microsoft.com/agent-framework/)): tarpinė programinė įranga, vykstanti aplink kiekvieną įrankio/funkcijos kvietimą, tinkama prieigos valdymo logikai ir patvirtinimo srautams.

Kiekvienas projektas trys subšablonus tvarko skirtingai: LangChain juos apvynioja kaip įrankius, AutoGen naudoja agento vaidmenį, o Microsoft Agent Framework naudoja funkcijų kvietimo tarpinę programinę įrangą. Prieš pasirinkdami dizainą savo agentui, perskaitykite vieną ar du įgyvendinimus nuo pradžios iki pabaigos.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
